# Cloudflare BYOIP 장애 재구성

합성 프리픽스와 가상 BGP 망에서 대량 철회 차단기가 피해 범위를 어떻게 바꾸고 광고 간격이 수렴에 어떤 영향을 주는지 살펴봅니다.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/cloudflare_byoip_outage.ipynb)

Colab 런타임은 **CPU**를 선택하고 위에서부터 실행하세요. 첫 코드 셀의 `QUICK = True`는 축소 조건을 사용합니다.

In [ ]:
QUICK = False

from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime
from http.server import BaseHTTPRequestHandler, HTTPServer
import json
import random
from threading import Thread
from unicodedata import east_asian_width
from urllib.request import ProxyHandler, build_opener

SEED = 20260220
TOTAL = 4306


def width(value):
    return sum(2 if east_asian_width(c) in "WF" else 1 for c in str(value))


def table(headers, rows):
    cells = [list(map(str, row)) for row in [headers, *rows]]
    widths = [max(map(width, column)) for column in zip(*cells)]
    for row in cells:
        print("  ".join(v + " " * (w - width(v))
                        for v, w in zip(row, widths)).rstrip())


def verdict(ok):
    return "PASS" if ok else "FAIL"


def finish(ok, **data):
    print(f"판정 {verdict(ok)}")
    return dict(passed=ok, **data)

### 그래프 언어

아래 셀에서 `CHART_LANG`의 기본값을 고르세요. `ko`는 한국어, `en`은 영어, `both`는 각 그래프를 한국어와 영어 순서로 그립니다. 한글 글꼴은 필요할 때 내려받습니다.

In [ ]:
# Chart language and style
import os
import platform
import sys
from pathlib import Path
from urllib.request import urlopen

import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.text import Text

CHART_LANG = os.environ.get('CHART_LANG', 'ko')
CHART_COLORS = ('#2a78d6', '#eb6834', '#1baf7a')
CHART_TEXT = '#0b0b0b'
CHART_MUTED = '#52514e'
CHART_GRID = '#e4e3df'
CHART_BG = '#fcfcfb'
CHART_LIMIT = '#e34948'

in_colab = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ
chart_font = matplotlib.rcParamsDefault['font.family']
requested_language = CHART_LANG
if requested_language not in ('ko', 'en', 'both'):
    raise ValueError("CHART_LANG must be ko, en or both")
if requested_language == 'both':
    CHART_LANG = 'ko'
if CHART_LANG == 'ko':
    # NanumGothic is licensed under the SIL OFL 1.1.
    nanum = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    try:
        if not nanum.is_file():
            nanum = Path('fonts/NanumGothic-Regular.ttf')
            if not nanum.is_file():
                nanum.parent.mkdir(exist_ok=True)
                url = 'https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf'
                with urlopen(url, timeout=60) as response:
                    nanum.write_bytes(response.read())
        font_manager.fontManager.addfont(str(nanum))
        chart_font = font_manager.FontProperties(fname=str(nanum)).get_name()
    except Exception as exc:
        CHART_LANG = 'en'
        reason = ' '.join(str(exc).splitlines()) or type(exc).__name__
        print(f'차트 언어 en, 한글 글꼴 설정 실패, {reason}')
    else:
        print(f'차트 언어 ko, 글꼴 {chart_font}')
else:
    CHART_LANG = 'en'

matplotlib.rcParams.update({
    'figure.figsize': (8, 4.5),
    'figure.dpi': 200,
    'savefig.dpi': 200,
    'figure.facecolor': CHART_BG,
    'savefig.facecolor': CHART_BG,
    'axes.facecolor': CHART_BG,
    'axes.edgecolor': CHART_MUTED,
    'axes.labelcolor': CHART_TEXT,
    'axes.titlelocation': 'left',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.axisbelow': True,
    'axes.unicode_minus': False,
    'axes.prop_cycle': matplotlib.rcsetup.cycler(color=CHART_COLORS),
    'axes.grid': True,
    'axes.grid.axis': 'y',
    'grid.color': CHART_GRID,
    'grid.linestyle': '-',
    'grid.linewidth': 0.8,
    'text.color': CHART_TEXT,
    'xtick.color': CHART_MUTED,
    'ytick.color': CHART_MUTED,
    'lines.linewidth': 2,
    'lines.markersize': 6,
    'legend.frameon': False,
    'font.family': chart_font,
})

chart_location = 'Colab' if in_colab else platform.system()
CPU_RUNTIME = f'{chart_location} CPU ({platform.machine()}), Python {platform.python_version()}'


def finish_chart(fig, runtime=CPU_RUNTIME, top=1):
    for text in fig.findobj(Text):
        label = text.get_text()
        if '·' in label:
            text.set_text(label.replace(' ·', ',').replace('·', ','))
    runtime = runtime.replace(' ·', ',').replace('·', ',')
    fig.tight_layout(rect=(0, 0.065, 1, top))
    fig.text(0.02, 0.025, runtime, fontsize=8, color=CHART_MUTED)
    plt.show()
if requested_language == 'both' and CHART_LANG == 'ko':
    CHART_LANG = 'both'
chart_languages = ('ko', 'en') if CHART_LANG == 'both' else (CHART_LANG,)

### 1. 값 없는 쿼리를 세 서버가 해석하는 방식

Cloudflare의 BYOIP 장애 사후 분석에 따르면, 삭제 대기 프리픽스를 조회하는 작업은 `pending_delete` 키만 보냈습니다. Go의 `Query().Get`은 키가 없을 때와 값이 비었을 때 모두 빈 문자열을 반환합니다. 원래 서버는 두 경우 모두 전체 목록을 선택했습니다.

아래 표는 쿼리 다섯 가지를 원래 서버, 키 존재 여부를 확인하는 수정안, 명시적인 불리언 값을 요구하는 엄격안에 적용합니다. 각 칸의 `FAIL`은 해당 응답이 표의 의도와 다르다는 뜻입니다.

In [ ]:
from urllib.parse import parse_qs, urlsplit

KEY = "pending_delete"
ALL, PENDING, BAD = "전체", "대기", "400"

def classify(query, policy):
    values = parse_qs(query, keep_blank_values=True)
    value = values.get(KEY, [""])[0]
    if policy == "original":
        return PENDING if value != "" else ALL
    if policy == "presence":
        return PENDING if KEY in values else ALL
    if policy == "strict":
        if KEY not in values:
            return ALL
        return {"true": PENDING, "false": ALL}.get(value.lower(), BAD)
    raise ValueError("알 수 없는 쿼리 정책")


def experiment1():
    print("## 실험1 쿼리 의미 표")
    print("대기는 삭제 대기분, 전체는 모든 프리픽스다.")
    print("의도 가정은 값 생략·빈 값·true가 대기, false가 전체다.")
    print("칸의 FAIL은 의도와 다른 응답을 뜻한다.")
    policies = ("original", "presence", "strict")
    cases = [("", ALL), (KEY, PENDING), (KEY + "=", PENDING),
             (KEY + "=true", PENDING), (KEY + "=false", ALL)]
    expected_misses = {
        (KEY, "original"), (KEY + "=", "original"),
        (KEY + "=false", "original"), (KEY + "=false", "presence"),
        (KEY, "strict"), (KEY + "=", "strict"),
    }
    misses = set()
    records, rows = [], []
    for query, intent in cases:
        responses = {p: classify(query, p) for p in policies}
        records.append(dict(query=query, intent=intent, **responses))
        cells = []
        for policy, response in responses.items():
            if response != intent:
                misses.add((query, policy))
                cells.append(response + " FAIL")
            else:
                cells.append(response)
        rows.append((query or '""', intent, *cells))
    table(("쿼리", "의도", "원본", "키 확인", "엄격"), rows)
    strict_ok = all(classify(q, "strict") == BAD
                    for q in (KEY, KEY + "=", KEY + "=unknown"))
    ok = misses == expected_misses and strict_ok
    print("실험 판정은 예상한 불일치와 400을 재현했는지 본다.")
    return finish(ok, cases=records)

query_results = experiment1()

### 2. 값 없는 HTTP 요청과 반환 개수

사후 분석에서 문제가 된 `GET /v1/prefixes?pending_delete` 요청을 두 로컬 HTTP 서버에 보냅니다. 기본 설정의 합성 프리픽스 4,306개 중 삭제 대기는 12개입니다. 원래 서버가 전체 4,306개를 반환하고 키를 확인하는 서버가 대기 중인 12개만 반환하는지 ID 목록까지 대조합니다. 각 서버는 요청 처리가 끝나면 종료됩니다.

In [ ]:
def fetch_fixture(policy, ids, pending):
    class PrefixAPI(BaseHTTPRequestHandler):
        def do_GET(self):
            target = urlsplit(self.path)
            if target.path != "/v1/prefixes":
                self.send_error(404)
                return
            choice = classify(target.query, policy)
            records = pending if choice == PENDING else ids
            body = json.dumps(records).encode("ascii")
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body)))
            self.end_headers()
            self.wfile.write(body)

        def log_message(self, format, *args):
            pass

    with HTTPServer(("127.0.0.1", 0), PrefixAPI) as server:
        thread = Thread(target=server.serve_forever,
                        kwargs={"poll_interval": 0.05}, daemon=True)
        thread.start()
        try:
            url = f"http://127.0.0.1:{server.server_port}/v1/prefixes?{KEY}"
            opener = build_opener(ProxyHandler({}))
            with opener.open(url, timeout=5) as response:
                return json.load(response)
        finally:
            server.shutdown()
            thread.join()


def experiment2(quick=False):
    print("## 실험2 값 없는 HTTP 요청")
    total, count = (430, 2) if quick else (TOTAL, 12)
    ids = list(range(total))
    pending = sorted(random.Random(SEED).sample(ids, count))
    print(f"가상 자료 {total:,}개 중 삭제 대기 {count}개")
    print("GET /v1/prefixes?pending_delete")
    original = fetch_fixture("original", ids, pending)
    fixed = fetch_fixture("presence", ids, pending)
    table(("서버", "반환 개수"), [("원본", len(original)), ("키 확인", len(fixed))])
    return finish(original == ids and fixed == pending, total=total,
                  pending=count, original=len(original), fixed=len(fixed))


http_results = experiment2(quick=QUICK)

### 3. 철회 배치와 회로 차단기

분마다 같은 크기의 철회 배치를 적용합니다. 최근 구간의 예상 철회량이 한도를 넘을 때 다음 배치 직전에 멈추는 정책과 계속 진행하는 정책을 비교합니다. 점선은 사건 규모의 참고값입니다.

In [ ]:
def withdrawal_curve(total, minutes, batch, window, percent=None):
    recent = deque()
    cumulative = [0]
    stopped = None
    for minute in range(1, minutes + 1):
        withdrawn = cumulative[-1]
        if stopped is None and withdrawn < total:
            while recent and recent[0][0] <= minute - window:
                recent.popleft()
            size = min(batch, total - withdrawn)
            proposed = sum(n for _, n in recent) + size
            if percent is not None and proposed * 100 > total * percent:
                stopped = minute
            else:
                recent.append((minute, size))
                withdrawn += size
        cumulative.append(withdrawn)
    return cumulative, stopped


def experiment3():
    print("## 실험3 철회 누적과 회로 차단기")
    print("가정은 17:56 UTC부터 1분마다 22개씩 50회다.")
    print("최근 5분 합계가 1%를 넘을 배치는 적용 전에 멈춘다.")
    batch, minutes, window = 22, 50, 5
    baseline, _ = withdrawal_curve(TOTAL, minutes, batch, window)
    guarded, stopped = withdrawal_curve(TOTAL, minutes, batch, window, 1)
    print("경과 분     차단기 없는 누적 개수 (왼쪽부터 매분)")
    print(" 0          0")
    for start in range(1, minutes + 1, 5):
        values = " ".join(f"{v:4}" for v in baseline[start:start + 5])
        print(f"{start:2}~{start + 4:2}분   {values}")
    print(f"차단기 적용 시 {stopped}분째 정지, 최종 {guarded[-1]}개")
    ok = (baseline == [batch * m for m in range(minutes + 1)]
          and stopped == 2 and guarded == [0] + [batch] * minutes)
    return finish(ok, minutes=list(range(minutes + 1)), baseline=baseline,
                  guarded=guarded, stopped_minute=stopped, batch=batch,
                  window_minutes=window, limit_percent=1, total=TOTAL)


withdrawal = experiment3()

In [ ]:
labels = {
    'ko': {
        'withdraw': '시간에 따른 누적 철회 프리픽스',
        'min': '경과 시간 (분)',
        'prefix': '누적 철회 (개)',
        'off': '차단기 없음',
        'on': '차단기 있음',
        'w_note': '분당 {b}개, {w}분 한도 {p}%, 점선은 약 1,100개',
        'as': '자율 시스템 수 (n)',
        'convergence_seconds': 'AS 수에 따른 BGP 수렴 시간',
        'time': '수렴 시간 (초)',
        'messages': 'AS 수에 따른 BGP 갱신 메시지 수',
        'count': '이웃별 갱신 메시지 (건)',
        'mrai_on': 'MRAI 켬 (30초)',
        'mrai_off': 'MRAI 끔',
        'bgp_note': '완전 연결망, 링크 지연 1초, 철회에도 MRAI 적용',
    },
    'en': {
        'withdraw': 'Cumulative prefix withdrawals',
        'min': 'Elapsed time (min)',
        'prefix': 'Prefixes withdrawn',
        'off': 'No breaker',
        'on': 'With breaker',
        'w_note': '{b} per minute, {p}% limit over {w} min, dashed line at about 1,100',
        'as': 'Autonomous systems (n)',
        'convergence_seconds': 'BGP convergence time by AS count',
        'time': 'Time to converge (s)',
        'messages': 'BGP update messages by AS count',
        'count': 'Update messages across peers',
        'mrai_on': 'MRAI on (30 s)',
        'mrai_off': 'MRAI off',
        'bgp_note': 'Full mesh, 1 s link delay, MRAI also applies to withdrawals',
    },
}

In [ ]:
for lang in chart_languages:
    t = labels[lang]
    fig, ax = plt.subplots()
    for key, label in (("baseline", "off"), ("guarded", "on")):
        ax.step(withdrawal["minutes"], withdrawal[key], where="post", label=t[label])
    ax.axhline(1100, color=CHART_LIMIT, linestyle="--", linewidth=1)
    ax.set(title=t["withdraw"], xlabel=t["min"], ylabel=t["prefix"])
    ax.legend()
    note = t["w_note"].format(b=withdrawal["batch"], w=withdrawal["window_minutes"],
                              p=withdrawal["limit_percent"])
    finish_chart(fig, f"{CPU_RUNTIME}, {note}")

### 4. BGP 경로 탐색과 최소 광고 간격(MRAI)

완전 연결망에서 원점 경로를 철회합니다. 같은 AS 수에서 광고 간격만 바꿔 모든 경로가 사라진 시각과 이웃에게 보낸 갱신 건수를 셉니다.

In [ ]:
@dataclass
class Tick:
    # 한 송신자의 광고는 1초 뒤 모든 이웃에 동시에 도착한다.
    arrivals: dict = field(default_factory=dict)
    timers: set = field(default_factory=set)


def path_hunt(n, mrai):
    nodes = range(n)
    peers = range(1, n)
    ordering = list(nodes)
    random.Random(SEED + n).shuffle(ordering)
    rank = {node: i for i, node in enumerate(ordering)}
    # 철회 직전 정상 상태. 이웃별 마지막 수신 경로를 보관한다.
    chosen = [(0,)] + [(i, 0) for i in peers]
    advertised = chosen.copy()
    received = [{j: chosen[j] for j in nodes if j != i} for i in nodes]
    traces = [[(0, path)] for path in chosen]
    last_sent = [-mrai] * n
    chosen[0] = None
    traces[0].append((0, None))
    calendar = defaultdict(Tick)
    calendar[0].timers.add(0)
    messages = 0
    all_down_since = None

    while calendar:
        now = min(calendar)
        tick = calendar.pop(now)
        # 같은 시각의 수신을 모두 반영한 뒤 한 번만 선택한다.
        if tick.arrivals:
            for i in peers:
                for sender, path in tick.arrivals.items():
                    if sender == i:
                        continue
                    if path is None or i in path:
                        received[i].pop(sender, None)
                    else:
                        received[i][sender] = path
                best = min(received[i].values(), default=None,
                           key=lambda p: (len(p), rank[p[0]]))
                route = None if best is None else (i,) + best
                if route != chosen[i]:
                    chosen[i] = route
                    traces[i].append((now, route))
                    tick.timers.add(i)

        for i in sorted(tick.timers):
            if chosen[i] == advertised[i]:
                continue
            due = last_sent[i] + mrai
            if now < due:
                calendar[due].timers.add(i)
                continue
            # 타이머 동안 바뀐 경로 중 현재 선택만 전파한다.
            advertised[i] = chosen[i]
            last_sent[i] = now
            calendar[now + 1].arrivals[i] = chosen[i]
            messages += n - 1
        if any(chosen):
            all_down_since = None
        elif all_down_since is None:
            all_down_since = now

    return dict(n=n, mrai_seconds=mrai, convergence_seconds=all_down_since,
                last_event_seconds=now, messages=messages, traces=traces,
                drained=all_down_since is not None and not any(advertised)
                and all(not received[i] for i in peers))


def experiment4():
    print("## 실험4 BGP 경로 사냥")
    print("완전 그래프, AS 0 원점, 링크 지연 1초, 동률은 고정 순위.")
    print("초기 광고는 안정 상태다. 첫 철회는 0초에 보낸다.")
    print("이 모형은 철회 메시지에도 같은 MRAI를 적용한다.")
    print("수렴은 모두가 마지막으로 경로를 잃은 시각이다.")
    print("메시지는 마지막 전파까지 이웃별로 센다.")
    runs, rows = [], []
    for n in range(4, 11):
        off, on = [path_hunt(n, mrai) for mrai in (0, 30)]
        rows.append((n, off["convergence_seconds"], off["messages"],
                     on["convergence_seconds"], on["messages"]))
        runs.extend((off, on))
    table(("AS 수", "끔 수렴(초)", "끔 전송(건)", "켬 수렴(초)", "켬 전송(건)"), rows)

    sample = next(r for r in runs if r["n"] == 6 and r["mrai_seconds"] == 30)
    traces = sample["traces"]
    witness = max(range(1, 6), key=lambda i: len(traces[i]))
    print(f"AS 6개, MRAI 30초에서 AS {witness}의 선택 경로")
    for time_s, path in traces[witness]:
        route = "경로 없음" if path is None else " → ".join(map(str, path))
        print(f"{time_s:3}초  {route}")
    print("0초의 직접 경로 뒤에 고른 우회로는 이미 철회된 길이다.")
    ok = (all(r["drained"] and r["messages"] >= r["n"] - 1 for r in runs)
          and len(traces[witness]) >= 4
          and all(path is None or (path[-1] == 0 and len(path) == len(set(path)))
                  for run in runs for trace in run["traces"] for _, path in trace))
    return finish(ok, runs=runs, witness_as=witness)


bgp = experiment4()

In [ ]:
for metric, unit in (("convergence_seconds", "time"), ("messages", "count")):
    for lang in chart_languages:
        t = labels[lang]
        fig, ax = plt.subplots()
        for interval, key, marker in ((30, "mrai_on", "o"), (0, "mrai_off", "x")):
            rows = [r for r in bgp["runs"] if r["mrai_seconds"] == interval]
            ax.plot([r["n"] for r in rows], [r[metric] for r in rows],
                    marker=marker, label=t[key])
        ax.set(title=t[metric], xlabel=t["as"], ylabel=t[unit])
        ax.set_xticks(sorted({r["n"] for r in bgp["runs"]}))
        ax.legend()
        finish_chart(fig, f"{CPU_RUNTIME}, {t['bgp_note']}")

### 5. 복구 시간

작업 정지를 기준으로 사건 타임라인의 경과 시간을 계산합니다. 복구 방식별 프리픽스 분할은 원본의 모형 가정입니다.

In [ ]:
def elapsed_minutes(start, end):
    fmt = "%H:%M"
    delta = datetime.strptime(end, fmt) - datetime.strptime(start, fmt)
    return int(delta.total_seconds() // 60)


def experiment5():
    print("## 실험5 복구 시간 모형")
    timeline = [
        ("18:46", "원인 작업 정지"),
        ("19:19", "대시보드 안내"),
        ("20:30", "바인딩이 남은 프리픽스 복구"),
        ("21:08", "전역 설정 적용 시작"),
        ("23:03", "수동 복구 완료"),
    ]
    start = timeline[0][0]
    events = [dict(utc=t, event=event, minutes=elapsed_minutes(start, t))
              for t, event in timeline]
    table(("UTC", "정지 후(분)", "공식 타임라인"),
          [(e["utc"], e["minutes"], e["event"]) for e in events])
    print("약 1,100개를 모형에서 800·200·100개로 둔다.")
    print("수동 복구 약 300개의 200·100 분할은 가정이다.")
    rollback = elapsed_minutes(start, "20:30")
    manual_end = elapsed_minutes(start, "23:03")
    apply_window = elapsed_minutes("21:08", "23:03")
    states = [
        dict(state="광고만 철회", count=800, method="되돌림", by=rollback),
        dict(state="일부 바인딩 삭제", count=200, method="수동 복구", by=manual_end),
        dict(state="전체 바인딩 삭제", count=100, method="수동 복구", by=manual_end),
    ]
    table(("초기 상태", "개수", "복구 방법", "정지 후 완료까지"),
          [(s["state"], s["count"], s["method"], f'{s["by"]}분 이내')
           for s in states])
    print(f"전역 설정 시작부터 완료까지 {apply_window}분이 걸렸다.")
    print("이 타임라인에서 수동 두 그룹의 개별 소요는 알 수 없다.")
    ok = (sum(s["count"] for s in states) == 1100
          and [e["minutes"] for e in events] == [0, 33, 104, 142, 257]
          and apply_window == 115)
    return finish(ok, timeline=events, states=states,
                  global_apply_minutes=apply_window)


recovery = experiment5()